# F2 — Preprocesamiento, limpieza, transformación y validación
**Proyecto:** Análisis del desempeño SIMCE 2025 — 4° Básico, Matemática

## Objetivo
Construir un pipeline **reproducible, modular, auditable y trazable** que:

- obtenga la base SIMCE 2025 desde `data/raw/`;
- explore el conjunto **antes de modificarlo**;
- limpie y estandarice tipos sin imputar información institucional o resultados;
- identifique y documente registros **Efectivos** y **No Efectivos**;
- incorpore `DimGeografia` **dentro del código**, sin depender de un CSV externo;
- valide casos normales, límites y excepciones;
- genere el dataset procesado para F3;
- genere evidencia del filtro (`auditoria_filtro.csv`) y de cobertura territorial (`cobertura_regional.csv`).

### Alcance de F2
Esta fase prepara y valida datos. **No realiza visualizaciones, pruebas de significancia, modelos predictivos ni priorización territorial**; esas tareas corresponden a fases posteriores.

### Fuente
Base pública SIMCE 2025 de la Agencia de Calidad de la Educación. El archivo bruto debe mantenerse sin modificaciones en `data/raw/`.

## 1. Configuración y reproducibilidad

Se utilizan rutas relativas al repositorio. El archivo procesado principal incorpora una marca de ejecución `AAAAMMDDHHMM`.  
La celda también muestra las versiones del entorno para dejar evidencia reproducible.

> Antes de entregar: ejecutar **Restart Kernel → Run All Cells** y conservar las salidas visibles.

In [1]:
from pathlib import Path
import platform
import sys
import pandas as pd
from IPython.display import display

# Permite ejecutar el notebook tanto desde la raíz como desde F2/.
PROYECTO_DIR = Path.cwd()
if not (PROYECTO_DIR / "src").exists():
    PROYECTO_DIR = PROYECTO_DIR.parent
if str(PROYECTO_DIR) not in sys.path:
    sys.path.insert(0, str(PROYECTO_DIR))

from src.configuracion import (
    encontrar_raw_dir, buscar_simce, obtener_fecha_ejecucion, COLUMNAS_SIMCE
)
from src.carga import construir_dim_geografia, cargar_simce
from src.diagnostico import diagnosticar_inicial
from src.transformacion import (
    convertir_tipos, limpiar_textos, agregar_categorias, agregar_geografia,
    clasificar_efectividad, construir_dataset_final, construir_cobertura_regional,
)
from src.auditoria import construir_auditoria
from src.validacion import validar_dataset_final
from src.exportacion import exportar_resultados

RAW_DIR = encontrar_raw_dir()
PROCESSED_DIR = RAW_DIR.parent / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SIMCE_FILE = buscar_simce(RAW_DIR)
FECHA_EJECUCION = obtener_fecha_ejecucion()
MARCA_EJECUCION = FECHA_EJECUCION.strftime("%Y%m%d%H%M")
OUTPUT_FILE = PROCESSED_DIR / f"simce4b2025_matematica_efectiva_{MARCA_EJECUCION}.csv"
AUDITORIA_FILE = PROCESSED_DIR / f"auditoria_filtro_{MARCA_EJECUCION}.csv"
COBERTURA_FILE = PROCESSED_DIR / f"cobertura_regional_{MARCA_EJECUCION}.csv"

print("=" * 72)
print("CONFIGURACIÓN")
print("=" * 72)
print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("Sistema:", platform.platform())
print("Entrada:", SIMCE_FILE)
print("Salida principal:", OUTPUT_FILE.name)
print("Auditoría:", AUDITORIA_FILE.name)
print("Cobertura:", COBERTURA_FILE.name)


CONFIGURACIÓN
Python : 3.12.6
Pandas : 3.0.5
Sistema: Windows-11-10.0.26200-SP0
Entrada: C:\Users\crist\f1_s01_evaluacion_entregable_grupo7\data\raw\simce4b2025_rbd_final.csv
Salida principal: simce4b2025_matematica_efectiva_202609222329.csv
Auditoría: auditoria_filtro_202609222329.csv
Cobertura: cobertura_regional_202609222329.csv


## 2. Catálogos y regla de efectividad

Los códigos administrativos se traducen a etiquetas comprensibles sin borrar las claves originales.

### Observaciones al puntaje
El proyecto utiliza el catálogo `OBS_PUNTAJE` para interpretar `marca_mate4b_rbd`. La variable representa una observación asociada a la posibilidad o representatividad de reportar el resultado.

**Punto de control importante sobre la marca 2:** existen casos en los que `marca_mate4b_rbd = 2` puede coexistir con un puntaje numérico. Por ello, F2 **no oculta esos casos**: los contabiliza, los deja explícitos en `auditoria_filtro.csv` y muestra una advertencia.

La glosa utilizada por el proyecto para el código 2 es *“Por causas ajenas a la Agencia, los resultados no son representativos del desempeño de los estudiantes”*. Antes de la entrega definitiva, el equipo debe conservar como respaldo el diccionario/glosa oficial de la **base 2025 específica**. El notebook no presenta como “confirmada 2025” una glosa que no esté respaldada por ese documento.

La regla operacional heredada del flujo del proyecto es:

- alumnos evaluados `> 0` **y**
- ausencia de observación al puntaje

→ **Efectiva**.

Todo lo demás queda como **No Efectiva** y se audita antes de excluirse.

In [2]:
dim_geo = construir_dim_geografia()

print(f"DimGeografia validada: {len(dim_geo):,} comunas")

display(dim_geo.head())


DimGeografia validada: 346 comunas


,cod_reg_rbd,region,cod_pro_rbd,provincia,cod_com_rbd,comuna,Zona,Macrozona,Orden,pais,ubicacion_region,ubicacion_provincia,ubicacion_comuna
0,1,Tarapacá,11,Iquique,1101,Iquique,Norte,Macrozona Norte,2,Chile,"Tarapacá, Chile","Iquique, Tarapacá, Chile","Iquique, Iquique, Tarapacá, Chile"
1,1,Tarapacá,11,Iquique,1107,Alto Hospicio,Norte,Macrozona Norte,2,Chile,"Tarapacá, Chile","Iquique, Tarapacá, Chile","Alto Hospicio, Iquique, Tarapacá, Chile"
2,1,Tarapacá,14,Tamarugal,1401,Pozo Almonte,Norte,Macrozona Norte,2,Chile,"Tarapacá, Chile","Tamarugal, Tarapacá, Chile","Pozo Almonte, Tamarugal, Tarapacá, Chile"
3,1,Tarapacá,14,Tamarugal,1402,Camiña,Norte,Macrozona Norte,2,Chile,"Tarapacá, Chile","Tamarugal, Tarapacá, Chile","Camiña, Tamarugal, Tarapacá, Chile"
4,1,Tarapacá,14,Tamarugal,1403,Colchane,Norte,Macrozona Norte,2,Chile,"Tarapacá, Chile","Tamarugal, Tarapacá, Chile","Colchane, Tamarugal, Tarapacá, Chile"


## 3. Obtención de datos

Para mantener el pipeline eficiente solo se cargan las variables requeridas por F2 y F3.  
Se eliminaron del flujo las variables de estándares de aprendizaje (`palu_eda_*`) porque se decidió que **no forman parte del producto analítico final**.

No se modifica el archivo original.

In [3]:
df_raw = cargar_simce(SIMCE_FILE, COLUMNAS_SIMCE)

print(f"Filas cargadas   : {df_raw.shape[0]:,}")

print(f"Columnas usadas  : {df_raw.shape[1]}")

print("Archivo bruto preservado: sí")


Filas cargadas   : 7,143
Columnas usadas  : 20
Archivo bruto preservado: sí


## 4. Exploración inicial — medir antes de decidir

La rúbrica exige que las decisiones de limpieza estén precedidas por evidencia.  
Por eso esta etapa se ejecuta **antes** de modificar tipos o filtrar registros.

Se revisan:

- dimensiones;
- tipos de datos;
- valores nulos;
- duplicados;
- cardinalidad;
- distribución básica de alumnos y puntaje;
- valores potencialmente atípicos mediante IQR.

Los atípicos **no se eliminan automáticamente**: un puntaje alto/bajo o un establecimiento con muchos alumnos puede ser un dato válido. El diagnóstico solo deja evidencia para justificar decisiones posteriores.

In [4]:
diagnostico_inicial = diagnosticar_inicial(df_raw)


DIAGNÓSTICO INICIAL
Dimensiones: 7,143 filas x 20 columnas
Filas duplicadas completas: 0
RBD repetidos: 0
Potenciales atípicos IQR (solo diagnóstico): {'nalu_mate4b_rbd': 303, 'prom_mate4b_rbd': 56}


,tipo_original,nulos,unicos
agno,int64,0,1
cod_com_rbd,int64,0,344
cod_depe1,int64,0,6
cod_depe2,int64,0,4
cod_deprov_rbd,int64,0,44
cod_grupo,float64,114,5
cod_pro_rbd,int64,0,56
cod_reg_rbd,int64,0,16
cod_rural_rbd,int64,0,2
codigo_bbdd,str,0,1


,nalu_mate4b_rbd,prom_mate4b_rbd
count,7143.000000,6579.000000
mean,29.954781,254.734762
std,29.617364,25.615483
min,0.000000,152.000000
25%,8.000000,237.000000
50%,22.000000,254.000000
75%,41.000000,272.000000
max,273.000000,341.000000


## 5. Estrategia de limpieza y tratamiento de nulos

### Decisiones
1. **No se imputan RBD, claves geográficas ni categorías institucionales.** Imputarlas inventaría identidad o pertenencia territorial.
2. **No se imputa `prom_mate4b_rbd`.** Un puntaje faltante no se reemplaza por media/mediana, porque alteraría el resultado educativo.
3. **No se imputa `nalu_mate4b_rbd`.** Los nulos se interpretan de forma conservadora como ausencia de alumnos que rinden la evaluación dentro del establecimiento, para la regla de efectividad.
4. Los textos se limpian eliminando caracteres de control, espacios repetidos y cadenas vacías.
5. Las variables numéricas se convierten de forma explícita. Si aparece texto inesperado en una columna numérica, el pipeline se detiene.
6. `fecha_bbdd` se convierte de formato `AAAAMMDD` a formato fecha real `AAAA-MM-DD`.
7. No se aplica escalamiento ni one-hot encoding en F2: esas transformaciones dependen del algoritmo que se seleccione en F3. Mantener las variables interpretables evita introducir una decisión de modelamiento antes de tiempo.

In [5]:
df_limpio = limpiar_textos(convertir_tipos(df_raw))

print("Tipos y textos normalizados.")

print("Fecha(s) de base:", df_limpio["fecha_bbdd"].dt.strftime("%Y-%m-%d").unique().tolist())


Tipos y textos normalizados.
Fecha(s) de base: ['2026-06-22']


## 6. Transformación: categorías, geografía y efectividad

Esta etapa:

- traduce dependencia, GSE y ruralidad;
- cruza las claves geográficas con la `DimGeografia` embebida;
- determina efectividad;
- conserva la causa de exclusión **antes** de filtrar;
- evita cambios silenciosos en el número de filas durante el `merge`.

Los nombres de región, provincia y comuna finales provienen de `DimGeografia`, no de nombres libres del archivo SIMCE.

In [6]:
df_transformado = agregar_categorias(df_limpio)

df_transformado = agregar_geografia(df_transformado, dim_geo)

df_transformado = clasificar_efectividad(df_transformado)

print(df_transformado["efectividad"].value_counts(dropna=False))


efectividad
Efectiva       6524
No Efectiva     619
Name: count, dtype: int64


## 7. Auditoría del filtro y análisis de materialidad

Antes de aplicar el filtro definitivo sobre los registros SIMCE, se realiza una auditoría del conjunto de datos con el propósito de **cuantificar el efecto de las exclusiones, justificar las decisiones adoptadas y mantener trazabilidad sobre los registros que no forman parte de la base analítica final**.

Los registros clasificados como **No Efectivos no son eliminados de manera silenciosa**. Previamente se identifican y caracterizan para evaluar las razones de su exclusión y su posible incidencia sobre el conjunto de datos.

La auditoría considera, entre otros, los siguientes elementos:

* cantidad de registros originales;
* cantidad de registros Efectivos y No Efectivos;
* registros que poseen puntaje disponible;
* registros No Efectivos que conservan puntaje;
* cantidad de estudiantes asociados al conjunto original y a los registros excluidos;
* promedio del puntaje de los registros Efectivos;
* promedio considerando todos los registros que poseen puntaje;
* diferencia entre ambos promedios;
* distribución de los registros según marca y motivo de exclusión;
* distribución territorial de los registros excluidos.

### 7.1 Criterio de conservación y exclusión

La disponibilidad de un dato no se considera, por sí sola, suficiente para determinar su utilidad analítica.

Para formar parte del conjunto procesado, un registro debe cumplir los criterios definidos de **efectividad y representatividad**. En consecuencia, un establecimiento puede disponer de un puntaje registrado y, aun así, ser excluido cuando la marca asociada indica que dicho resultado no representa adecuadamente el desempeño que se busca estudiar.

Por esta razón, el criterio de exclusión no se basa únicamente en la existencia o ausencia de `prom_mate4b_rbd`, sino también en la información contenida en `marca_mate4b_rbd` y su correspondiente descripción normalizada.

Los registros clasificados como No Efectivos se conservan dentro de la auditoría para mantener su trazabilidad, pero **no se incorporan al dataset analítico final**, debido a que sus condiciones de aplicación o representatividad reducen su utilidad para responder la problemática definida en el proyecto.

### 7.2 Análisis de materialidad

El análisis de materialidad busca determinar la relevancia que tienen los registros excluidos respecto del conjunto original, sin utilizar un porcentaje arbitrario como umbral para decidir si una exclusión es importante o no.

La evaluación se realiza considerando de manera conjunta tres dimensiones:

1. **Cantidad absoluta de registros y estudiantes excluidos**, para dimensionar el volumen de información afectada.
2. **Diferencia entre los promedios de puntaje**, comparando los registros Efectivos con el conjunto total de registros que poseen puntaje.
3. **Concentración territorial de las exclusiones**, para identificar si los registros descartados se distribuyen de manera generalizada o se concentran en determinadas regiones o territorios.

La diferencia entre el promedio de los registros Efectivos y el promedio de todos los registros con puntaje se utiliza como una **medida de sensibilidad del filtro**. Su objetivo es evaluar cuánto cambia el comportamiento general del puntaje al aplicar los criterios de representatividad.

Esta comparación no se utiliza como criterio para reincorporar registros excluidos. Un impacto reducido sobre el promedio general no convierte un resultado No Efectivo en representativo, del mismo modo que una diferencia mayor no implica automáticamente que el registro deba conservarse.

### 7.3 Tratamiento de registros con marca 2

Se presta especial atención a los registros con `marca_mate4b_rbd = 2`.

La normalización de las marcas permite interpretar estos casos como resultados que, por causas ajenas a la Agencia de Calidad de la Educación, **no son considerados representativos del desempeño de los estudiantes evaluados**.

En algunos de estos registros puede existir un valor numérico en `prom_mate4b_rbd`. Sin embargo, la existencia del puntaje no elimina la restricción asociada a su representatividad.

Para los objetivos del presente proyecto, cuyo análisis posterior requiere comparar resultados académicos entre establecimientos y características territoriales o socioeconómicas, incorporar observaciones identificadas explícitamente como no representativas podría introducir información cuya interpretación no es equivalente a la de los registros Efectivos.

Por esta razón, estos casos:

* permanecen identificados dentro de la auditoría;
* mantienen disponible su información original para fines de trazabilidad;
* no son imputados ni modificados;
* y son excluidos del conjunto analítico final.

De esta forma se distingue entre **disponibilidad del dato** y **validez del dato para el propósito específico del análisis**.

### 7.4 Decisión sobre valores faltantes e imputación

No se realiza imputación del puntaje promedio para los registros que no cuentan con un resultado válido.

Esta decisión responde a que la ausencia de información en estos casos no corresponde necesariamente a un problema técnico de pérdida de datos que pueda solucionarse mediante media, mediana u otro mecanismo de imputación.

En parte de los registros, la ausencia o invalidación del resultado está relacionada con condiciones de aplicación, número de estudiantes evaluados o restricciones de representatividad establecidas en la propia fuente.

Asignar artificialmente un puntaje a estos casos podría:

* introducir valores que no fueron observados;
* reducir artificialmente la variabilidad del conjunto;
* alterar comparaciones entre establecimientos;
* y otorgar validez analítica a registros que originalmente no cumplen los criterios de representatividad.

Por lo anterior, se opta por **no imputar puntajes** y mantener separados los registros que no cumplen los requisitos establecidos.

### 7.5 Componente territorial de la auditoría

Debido a que la problemática del proyecto considera diferencias territoriales en los resultados SIMCE, la auditoría también analiza la distribución regional de los registros excluidos.

Este control permite detectar si las exclusiones presentan una concentración territorial relevante. Esta revisión es importante porque una eliminación desproporcionada de establecimientos pertenecientes a una determinada región podría modificar la cobertura territorial del conjunto procesado y afectar posteriormente las comparaciones geográficas.

La dimensión territorial se utiliza, por tanto, como un **control de cobertura y representatividad del dataset procesado**, y no como un criterio independiente para reincorporar observaciones que no cumplen las condiciones de efectividad.

### 7.6 Decisión metodológica final

El filtrado aplicado busca priorizar la **calidad y representatividad de las observaciones por sobre la conservación indiscriminada de registros**.


In [7]:
auditoria_filtro, resumen_filtro, resumen_marcas = (
    construir_auditoria(
        df_transformado
    )
)

display(resumen_filtro)

display(resumen_marcas)

print(
    "Casos marca 2 con puntaje:",
    int(
        auditoria_filtro[
            "marca_2_con_puntaje"
        ].sum()
    )
)


,indicador,valor
0,Registros originales,7143.0000
1,Registros efectivos,6524.0000
2,Registros no efectivos,619.0000
3,Registros con puntaje,6579.0000
4,No efectivos con puntaje,55.0000
5,Alumnos asociados al bruto,213967.0000
6,Alumnos asociados a no efectivos,1299.0000
7,Promedio puntaje - efectivos,254.6971
8,Promedio puntaje - todos con puntaje,254.7348
9,Diferencia de promedio (puntos),0.0376


,marca_mate4b_rbd,obs_puntaje,registros,alumnos,con_puntaje
2,NaN,NaN,6638,212668,6524
0,1.0,"No es posible reportar resultados, porque la c...",450,439,0
1,2.0,"Por causas ajenas a la Agencia, los resultados...",55,860,55


Casos marca 2 con puntaje: 55


## 8. Pruebas técnicas — casos normal, límite y excepción

Las pruebas fueron trasladadas a `tests/test_regla_efectividad.py` para que sean **reejecutables fuera del notebook**. Esta celda ejecuta ese archivo y conserva la evidencia visible en F2.

Se verifican cinco escenarios:

- **Caso normal:** alumnos > 0, sin marca → Efectiva.
- **Caso límite:** 0 alumnos → No Efectiva.
- **Caso con observación:** alumnos > 0 y marca documentada → No Efectiva.
- **Caso marca 2 con puntaje:** debe quedar No Efectiva y ser auditable.
- **Excepción:** código de marca desconocido → el proceso debe detenerse.


In [8]:
import subprocess
import sys

tests_dir = PROJECT_ROOT / "tests"

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio de pruebas: {tests_dir}")
print(f"¿Existe tests?: {tests_dir.exists()}")

if not tests_dir.exists():
    raise FileNotFoundError(
        f"No se encontró la carpeta de pruebas: {tests_dir}"
    )

resultado_pruebas = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-p",
        "test_*.py",
        "-v",
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
)

# unittest normalmente escribe su salida en stderr
if resultado_pruebas.stdout:
    print(resultado_pruebas.stdout)

if resultado_pruebas.stderr:
    print(resultado_pruebas.stderr)

if resultado_pruebas.returncode != 0:
    raise RuntimeError(
        "Las pruebas técnicas de F2 no fueron superadas."
    )

print("Pruebas técnicas superadas correctamente.")


NameError: name 'PROJECT_ROOT' is not defined

## 9. Filtrado y estandarización del producto final

Solo después del diagnóstico y la auditoría se conservan registros **Efectivos**.

Las columnas auxiliares utilizadas para decidir el filtro no se necesitan en el dataset final porque quedan preservadas en `auditoria_filtro.csv`.

Las variables de estándares de aprendizaje porcentuales fueron eliminadas del F2 por decisión de alcance y **no se cargan ni transforman**.

In [ ]:
df_final = construir_dataset_final(df_transformado)

print(
    f"Dataset final construido: "
    f"{df_final.shape[0]:,} filas x {df_final.shape[1]} columnas"
)


Dataset final construido: 6,524 filas x 30 columnas


## 10. Cobertura regional

`cobertura_regional.csv` permite comprobar que el filtro no afecta de forma invisible a determinadas regiones.

Se informan **conteos absolutos** por región:

- registros originales;
- registros efectivos;
- registros no efectivos;
- excluidos que conservaban puntaje;
- alumnos del bruto;
- alumnos efectivos;
- alumnos asociados a no efectivos.

Este archivo es evidencia de F2; no constituye todavía una priorización territorial.

In [ ]:
cobertura_regional = construir_cobertura_regional(df_transformado)

display(cobertura_regional)


,cod_reg_rbd,region,Zona,Macrozona,Orden,registros_originales,registros_efectivos,registros_no_efectivos,no_efectivos_con_puntaje,alumnos_bruto,alumnos_efectivos,alumnos_no_efectivos
14,15,Arica y Parinacota,Norte,Macrozona Norte,1,81,69,12,1,3023,2946,77
0,1,Tarapacá,Norte,Macrozona Norte,2,109,102,7,0,5295,5290,5
1,2,Antofagasta,Norte,Macrozona Norte,3,136,134,2,0,8307,8305,2
2,3,Atacama,Norte,Macrozona Norte,4,110,102,8,0,4082,4076,6
3,4,Coquimbo,Norte,Macrozona Centro Norte,5,447,372,75,2,10187,10106,81
4,5,Valparaíso,Centro,Macrozona Centro Norte,6,759,740,19,3,21403,21361,42
12,13,Metropolitana de Santiago,Centro,Macrozona Metropolitana,7,1754,1723,31,16,82485,82209,276
5,6,Libertador General Bernardo O'Higgins,Centro,Macrozona Centro Norte,8,445,421,24,5,11885,11798,87
6,7,Maule,Centro,Macrozona Centro Sur,9,558,505,53,6,13615,13507,108
15,16,Ñuble,Centro,Macrozona Centro Sur,10,290,251,39,2,5495,5436,59


## 11. Validaciones finales de integridad y coherencia

Las validaciones comprueban:

- dataset no vacío;
- solo Matemática;
- solo registros Efectivos;
- alumnos evaluados mayores que cero;
- RBD no nulo y único;
- puntaje y geografía completos;
- códigos territoriales completos;
- fecha válida;
- ausencia de columnas duplicadas;
- reconciliación entre bruto, efectivos y auditoría;
- reconciliación de cobertura regional;
- ausencia de las variables porcentuales eliminadas del alcance.

In [ ]:
CRITERIOS_CUMPLIDOS = validar_dataset_final(
    df_final,
    df_transformado,
    auditoria_filtro,
    cobertura_regional,
)

print("=" * 72)

print("VALIDACIONES FINALES")

print("=" * 72)

for criterio, cumple in CRITERIOS_CUMPLIDOS.items():
    print(f"✓ {criterio}" if cumple else f"✗ {criterio}")


VALIDACIONES FINALES
✓ Dataset final no vacío
✓ Asignatura = Matematica
✓ Efectividad = Efectiva
✓ n_alumnos > 0
✓ RBD no nulo
✓ RBD único
✓ Puntaje no nulo
✓ Geografía completa
✓ fecha_bbdd válida
✓ Sin columnas duplicadas
✓ Reconciliación bruto = final + auditoría
✓ Cobertura regional reconcilia con bruto
✓ Cobertura regional reconcilia con final
✓ Sin columnas pct eliminadas


## 12. Exportación y verificación de ida y vuelta

Se generan tres productos reales:

1. `simce4b2025_matematica_efectiva_AAAAMMDDHHMM.csv`  
   Dataset limpio y validado para F3.
2. `auditoria_filtro.csv`  
   Registros No Efectivos y causa de exclusión.
3. `cobertura_regional.csv`  
   Reconciliación territorial antes/después del filtro.

Después de exportar el dataset principal, se vuelve a leer y se compara con el DataFrame original para detectar alteraciones de escritura.

In [ ]:
exportar_resultados(
    df_final,
    auditoria_filtro,
    cobertura_regional,
    OUTPUT_FILE,
    AUDITORIA_FILE,
    COBERTURA_FILE,
)

print("=" * 72)

print("RESULTADO F2")

print("=" * 72)

print(f"Filas originales              : {len(df_transformado):,}")

print(f"Filas válidas / efectivas     : {len(df_final):,}")

print(f"Filas no efectivas auditadas  : {len(auditoria_filtro):,}")

print(
    "No efectivas con puntaje      : "
    f"{int(auditoria_filtro['prom_mate4b_rbd'].notna().sum()):,}"
)

print(
    "Marca 2 con puntaje auditada  : "
    f"{int(auditoria_filtro['marca_2_con_puntaje'].sum()):,}"
)

print(f"Archivo principal             : {OUTPUT_FILE.name}")

print(f"Auditoría                     : {AUDITORIA_FILE.name}")

print(f"Cobertura regional            : {COBERTURA_FILE.name}")


RESULTADO F2
Filas originales              : 7,143
Filas válidas / efectivas     : 6,524
Filas no efectivas auditadas  : 619
No efectivas con puntaje      : 55
Marca 2 con puntaje auditada  : 55
Archivo principal             : simce4b2025_matematica_efectiva_202609212142.csv
Auditoría                     : auditoria_filtro_202609212142.csv
Cobertura regional            : cobertura_regional_202609212142.csv


## 13. Conclusión técnica para F2

La fase deja un conjunto de datos procesado **sin imputar puntajes, identidades ni territorio**. Las exclusiones se realizan únicamente después de medir y documentar sus causas.

La auditoría permite distinguir dos preguntas diferentes:

- **¿Cuántos establecimientos salen de la base?**
- **¿Cuántos de los excluidos todavía tenían puntaje?**

Esa distinción es importante para discutir materialidad sin asumir que todo registro excluido afecta de la misma forma el análisis posterior. La diferencia entre el promedio de efectivos y el promedio de todos los registros con puntaje se muestra como prueba de sensibilidad, pero **no se utiliza para reincorporar registros cuya representatividad está observada**.

Los casos `marca_mate4b_rbd = 2` con puntaje se preservan en la auditoría para que la decisión pueda verificarse contra el diccionario oficial 2025 antes de la entrega final.

### Tratamientos deliberadamente NO aplicados
- No se eliminan atípicos solo por IQR.
- No se imputa puntaje promedio.
- No se imputa geografía.
- No se escalan variables todavía.
- No se codifican categorías mediante one-hot todavía.
- No se realizan gráficos ni inferencia estadística en F2.

## 14. Vinculación con el mapa conceptual y fases del proyecto

| Elemento | Estado | Evidencia |
|---|---|---|
| Fuente oficial SIMCE 2025 | Implementado | `data/raw/` + carga selectiva |
| Exploración y calidad | Implementado en F2 | diagnóstico de tipos, nulos, duplicados y atípicos |
| Limpieza | Implementado en F2 | casting, fecha, limpieza de textos |
| Transformación | Implementado en F2 | categorías + `DimGeografia` embebida |
| Control de representatividad | Implementado en F2 | efectividad + `auditoria_filtro.csv` |
| Validación técnica | Implementado en F2 | asserts + pruebas normal/límite/excepción |
| Cobertura territorial | Implementado como control F2 | `cobertura_regional.csv` |
| Análisis estadístico / significancia | Proyectado | Fase 3 |
| Escalamiento/encoding para algoritmo | Proyectado | Fase 3, según modelo elegido |
| Visualización y comunicación de resultados | Proyectado | fase posterior |

Esta tabla evita afirmar que componentes posteriores ya están desarrollados y deja trazabilidad directa entre planificación e implementación.

## 15. Checklist previo a entrega

Antes de subir el notebook definitivo:

- ejecutar **Restart Kernel → Run All Cells**;
- comprobar que la numeración de ejecución sea continua;
- **no borrar los outputs**;
- verificar que `data/processed/` contenga los tres productos de F2;
- adjuntar/conservar respaldo del diccionario oficial SIMCE 2025 utilizado para `marca_mate4b_rbd`, especialmente el código 2;
- comprobar que README explique cómo ejecutar F1 y F2;
- comprobar que `requirements.txt` tenga versiones fijadas;
- citar en el informe las cifras producidas por `resumen_filtro`, no cifras escritas manualmente.